# R07-All — N01 Lesion-Aware Comprehensive Ensemble Benchmark (Seed 42)

**Kaggle notebook:** `lesion-r07-n01-all-ensembles-s42`. **Inputs:** completed Kaggle outputs for `lesion-r04-ssl-classification-s42`, `lesion-r05-n01-dual-ssl-featurebank-s42`, `lesion-r06-n01-rotation-simclr-feature-fusion-s42`. **GPU:** not required; **Internet:** off.

This benchmark reuses the exact six N01 classifiers and compares **single-model references, hard voting, arithmetic/geometric averaging, two-branch and all-branch averaging, SimCLR-dominant/fusion-dominant/random constrained weighted averaging, confidence/entropy/margin-gated averaging, multinomial stacking (raw/log/entropy-augmented features), anchored stacking, and small class-bias corrections**. Historical thesis families are reproduced in *method*, not their old (leaky) fitting protocol. Confidence/entropy gating is an additional Phase-2G-style extension; these are clearly labeled.

**Partition roles:** `train` = base-model fitting only; `meta_train` = probability calibration/weight/gate/meta-learner fitting and lesion-group OOF selection where feasible; `validation` = held-out comparison/locked configuration selection; `test` = **not accessed**. The model-selection leaderboard is exploratory and can be optimistic after comparing many candidates. Do not report validation as independent test results. Save the frozen configuration before a separate R08 evaluation. The original historical `SimCLR_SelectedTTA` is **not available** from the R05/R06 inputs; do not synthesize its predictions or claim it was evaluated here.

This notebook incorporates the rare-class group-fold correction from R07 v2: if all-class group CV is impossible, fall back to a predeclared stacking regularization, and never compute mean recall over empty classes.

## 1. Setup, strict source verification and image/lesion alignment

In [1]:
import os,json,hashlib,random,time,zipfile,shutil,io,math
from pathlib import Path
import numpy as np,pandas as pd,torch,joblib
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score,log_loss,confusion_matrix,classification_report

NAME='lesion-r07-n01-all-ensembles-s42';SEED=42
INPUT=Path(os.getenv('R07_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data/r07_test_inputs')).resolve()
OUT=Path(os.getenv('R07_OUT','/kaggle/working/'+NAME if Path('/kaggle/working').exists() else './'+NAME)).resolve()
STAGING=Path(os.getenv('R07_STAGE_DIR','/kaggle/working/.r07_input' if Path('/kaggle/working').exists() else './.r07_input')).resolve()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
SOURCES=['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
EXPECTED={'train':16490,'meta_train':2267,'validation':2186}
C_GRID=[0.01,0.1,1.0];STACK_ALPHA=[0.3,0.5,0.7,0.9]
BIAS_GRID=[-.15,-.075,0.,.075,.15];WEIGHT_SAMPLES=4096;FOLDS=5
MAX_HOURS=7.5;START=time.monotonic()
random.seed(SEED);np.random.seed(SEED)
OUT.mkdir(parents=True,exist_ok=True)
for sub in ['reports','tables','predictions','models']:(OUT/sub).mkdir(exist_ok=True)

def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def jwrite(p,x):
    p=Path(p);tmp=p.with_suffix('.tmp');tmp.write_text(json.dumps(x,indent=2,allow_nan=False,default=str));tmp.replace(p)
def normalize(p):
    p=np.asarray(p,dtype=np.float64)
    req(p.ndim==2 and p.shape[1]==8 and np.isfinite(p).all(),'Invalid source probability tensor')
    p=np.maximum(p,1e-12);return p/p.sum(axis=1,keepdims=True)
def score(y,p):
    pred=np.argmax(p,axis=1)
    return {'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'accuracy':float(accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
            'nll':float(log_loss(y,normalize(p),labels=np.arange(8)))}
def tscale(p,t):return softmax(np.log(np.clip(p,1e-12,1))/float(t),axis=1)
def apply_bias(p,b):return softmax(np.log(np.clip(p,1e-12,1))+np.asarray(b)[None,:],axis=1)
def calibrate(p,y):
    obj=lambda lt: log_loss(y,tscale(p,np.exp(lt)),labels=np.arange(8))
    r=minimize_scalar(obj,bounds=(np.log(.45),np.log(3.5)),method='bounded',options={'xatol':1e-4})
    req(r.success,'Temperature optimization did not converge')
    return float(np.exp(r.x))
print(NAME,'torch',torch.__version__,'input',INPUT)

lesion-r07-n01-all-ensembles-s42 torch 2.11.0+cpu input /kaggle/input


In [2]:
def discover(root):
    found={'R04':[],'R05':[],'R06':[]}
    markers={'R04':'reports/run_summary.json','R05':'reports/final_status.json','R06':'reports/final_status.json'}
    for kind,marker in markers.items():
        for p in root.rglob(Path(marker).name):
            if not p.as_posix().endswith(marker):continue
            try:d=json.loads(p.read_text())
            except Exception:continue
            if d.get('stage')!=kind or d.get('seed')!=SEED or d.get('status')!='COMPLETED':continue
            candidate=p.parent.parent
            found[kind].append(candidate.resolve())
        found[kind]=sorted(set(found[kind]))
    return found

def unpack(zp,dst):
    dst.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        files=[x for x in z.infolist() if not x.is_dir()]
        req(sum(x.file_size for x in files)<min(12*2**30,int(shutil.disk_usage(dst).free*.85)),'Insufficient disk to unpack input')
        for ent in files:
            rel=Path(ent.filename.replace('\\','/'))
            req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe archive path')
            path=(dst/rel).resolve();req(path.is_relative_to(dst.resolve()),'ZIP traversal')
            mode=(ent.external_attr>>16)&0o170000
            req(mode!=0o120000,'Symlink in input ZIP')
            path.parent.mkdir(exist_ok=True,parents=True)
            if path.exists() and path.stat().st_size==ent.file_size:continue
            with z.open(ent) as a,open(path,'wb') as b:shutil.copyfileobj(a,b,2**20)

roots=discover(INPUT)
if any(len(v)!=1 for v in roots.values()):
    archives=sorted(INPUT.rglob('*.zip'))
    for level in range(2):
        if all(len(v)==1 for v in roots.values()):break
        for zp in archives:
            dst=STAGING/hashlib.sha256(str(zp).encode()).hexdigest()[:12]
            if (dst/'.ok').exists():continue
            with zipfile.ZipFile(zp) as z:
                names=z.namelist()
                relevant=zp.name=='_output_.zip' or any(n.endswith(('reports/run_summary.json','reports/final_status.json')) for n in names) or any(n.lower().endswith('.zip') for n in names)
            if not relevant:continue
            unpack(zp,dst);(dst/'.ok').write_text('ok')
        candidates=[discover(INPUT),discover(STAGING)]
        roots={k:sorted(set(candidates[0][k]+candidates[1][k])) for k in roots}
        archives=sorted(set(archives+[p for p in STAGING.rglob('*.zip') if p.is_file()]))
for kind,v in roots.items():req(len(v)==1,f'Attach exactly one completed {kind} seed-42 Kaggle output; found: {v}')
R04,R05,R06=(roots[k][0] for k in ['R04','R05','R06'])
s04=json.loads((R04/'reports/run_summary.json').read_text())
s05=json.loads((R05/'reports/final_status.json').read_text())
s06=json.loads((R06/'reports/final_status.json').read_text())
req(s04.get('stage')=='R04' and s04.get('status')=='COMPLETED' and s04.get('seed')==SEED,'R04 not completed or wrong seed')
req(s05.get('stage')=='R05' and s05.get('status')=='COMPLETED' and s05.get('seed')==SEED,'R05 not completed or wrong seed')
req(s06.get('stage')=='R06' and s06.get('status')=='COMPLETED' and s06.get('seed')==SEED,'R06 not completed or wrong seed')
req(s06['r04_encoder_provenance']==s05['r04_provenance'],'R06/R05 R04 encoder provenance mismatch')
req(not s05['test_touched'] and not s06['test_touched'],'An input stage accessed the locked test')
req(s05['classes']==CLASSES and s05['splits']==EXPECTED,'Wrong class order or split sizes')
req(s04['n01_manifest_sha256']==s05['n01_manifest_sha256']==s06['n01_manifest_sha256'],'N01 cohort/hash mismatch')
req(s05['r04_provenance']['rotation']['encoder_sha256'] and s05['r04_provenance']['simclr']['encoder_sha256'],'Missing encoder provenance')
print('RESOLVED',{'R04':str(R04),'R05':str(R05),'R06':str(R06)})

RESOLVED {'R04': '/kaggle/input/notebooks/sudiptoroy007/lesion-r04-ssl-classification-s42/lesion-r04-ssl-classification-s42', 'R05': '/kaggle/input/notebooks/masumbillah029/lesion-r05-n01-dual-ssl-featurebank-s42/lesion-r05-n01-dual-ssl-featurebank-s42', 'R06': '/kaggle/input/notebooks/masumbillah029/lesion-r06-n01-rotation-simclr-feature-fusion-s42/lesion-r06-n01-rotation-simclr-feature-fusion-s42'}


In [3]:
meta={};truth={};prob={split:{} for split in ['meta_train','validation']}
for split,n in EXPECTED.items():
    item=s05['manifests'][split];f=R05/item['file']
    req(f.is_file() and sha(f)==item['sha256'],f'{split} feature manifest missing/corrupt')
    d=pd.read_csv(f,dtype='string',keep_default_na=False)
    req(len(d)==n and d.image_id.is_unique and d.label.isin(CLASSES).all(),f'Wrong {split} images/labels')
    req(d.class_index.eq(d.label.map({c:str(i) for i,c in enumerate(CLASSES)})).all(),'Class ordering mismatch')
    for key in ['image_id','group_id','source_sha256']:
        req(d[key].ne('').all(),f'{split} missing {key}')
    meta[split]=d;truth[split]=d.class_index.astype(int).to_numpy(dtype=np.int64)
for a,b in [('train','meta_train'),('train','validation'),('meta_train','validation')]:
    for k in ['image_id','group_id','source_sha256']:
        req(set(meta[a][k]).isdisjoint(set(meta[b][k])),f'{k}: overlap between {a}/{b}')
req(all(len(np.unique(truth[s]))==8 for s in ['meta_train','validation']),'Missing class in stacking/evaluation')

for source in ['rotation','simclr']:
    src=s05['r04_provenance'][source]
    ep=R04/'models'/source/'encoder_best.pt';bp=R04/'models'/source/'best.pt';vp=R04/'predictions'/source/'validation.npz'
    req(all(p.is_file() for p in [ep,bp,vp]),f'R04 {source} incomplete')
    req(sha(ep)==src['encoder_sha256'] and sha(bp)==src['best_classifier_sha256'] and sha(vp)==src['validation_prob_sha256'],f'{source} R04 hash/provenance failed')
    ck=torch.load(bp,map_location='cpu',weights_only=False)
    weights=ck['model']['classifier.weight'].detach().cpu().numpy().astype(np.float64)
    bias=ck['model']['classifier.bias'].detach().cpu().numpy().astype(np.float64)
    req(weights.shape==(8,2048) and bias.shape==(8,),f'{source} classifier head mismatch')
    for split in prob:
        item=s05['features'][split][source];p=R05/item['file']
        req(p.is_file() and sha(p)==item['sha256'],f'{split}/{source} feature file invalid')
        x=np.load(p,mmap_mode='r',allow_pickle=False)
        req(x.shape==(EXPECTED[split],2048) and np.isfinite(x).all(),f'{split}/{source} features invalid')
        prob[split][source]=softmax(np.asarray(x,dtype=np.float64)@weights.T+bias,axis=1)
    with np.load(vp,allow_pickle=False) as z:
        req(np.array_equal(z['image_id'].astype(str),meta['validation'].image_id.astype(str).to_numpy()),f'{source} R04 validation IDs mismatch')
        req(np.array_equal(z['y'].astype(int),truth['validation']),f'{source} R04 validation targets mismatch')
        req(np.array_equal(z['classes'].astype(str),CLASSES),f'{source} R04 class mismatch')
        diff=float(np.max(np.abs(prob['validation'][source]-normalize(z['proba']))))
        req(diff<.02,f'R04 {source} checkpoint reconstruction mismatch: {diff:.4f}')
    print('R04 VERIFIED',source,'validation BACC',round(score(truth['validation'],prob['validation'][source])['balanced_accuracy']*100,3),'max reconstruction difference',round(diff,6))

for source in SOURCES[2:]:
    for split in prob:
        p=R06/'predictions'/f'{source}_{split}.npz';req(p.is_file(),f'Missing {p}')
        with np.load(p,allow_pickle=False) as z:
            req(np.array_equal(z['classes'].astype(str),CLASSES),'R06 class mismatch')
            req(np.array_equal(z['image_id'].astype(str),meta[split].image_id.astype(str).to_numpy()),f'R06 {source} {split} image alignment failure')
            req(np.array_equal(z['group_id'].astype(str),meta[split].group_id.astype(str).to_numpy()),f'R06 {source} {split} group mismatch')
            req(np.array_equal(z['y'].astype(int),truth[split]),f'R06 {source} {split} label mismatch')
            req(str(z['model_id'].item())==source,f'Wrong R06 {source} model')
            probs=normalize(z['proba']);logits=np.asarray(z['logits'],dtype=np.float64)
            req(probs.shape==(EXPECTED[split],8) and logits.shape==probs.shape,'Incorrect R06 prediction dimension')
            req(np.max(abs(probs-softmax(logits,axis=1)))<.001,f'R06 {source} probabilities/logits disagree')
            prob[split][source]=probs
for split in prob:req(set(prob[split])==set(SOURCES),'Missing ensemble source')
print('PREFLIGHT PASSED: six aligned prediction sources; no test-set inputs read')

R04 VERIFIED rotation validation BACC 57.717 max reconstruction difference 0.001828
R04 VERIFIED simclr validation BACC 57.562 max reconstruction difference 0.001351
PREFLIGHT PASSED: six aligned prediction sources; no test-set inputs read


In [4]:
y_meta=truth['meta_train'];y_val=truth['validation']
g_meta=meta['meta_train'].group_id.astype(str).to_numpy()
g_val=meta['validation'].group_id.astype(str).to_numpy()

def group_support(y,g):
    return {CLASSES[c]:int(len(np.unique(g[y==c]))) for c in range(8)}

def safe_group_cv(y,g,max_folds=5,seed=42):
    y=np.asarray(y,dtype=np.int64);g=np.asarray(g,dtype=str)
    req(len(y)==len(g) and len(np.unique(y))==8,'Invalid grouped-CV input')
    support=group_support(y,g);k=min(max_folds,*support.values())
    if k<2:
        return [],{'mode':'NO_CLASS_COMPLETE_GROUP_CV','folds':0,'group_support':support,
                   'reason':'A class has fewer than two distinct lesion groups; full-class grouped CV is impossible.'}
    groups,inv=np.unique(g,return_inverse=True)
    labels=np.full(len(groups),-1,dtype=np.int64)
    sizes=np.bincount(inv)
    pure=True
    for j in range(len(groups)):
        unique=np.unique(y[inv==j])
        if len(unique)!=1:pure=False;break
        labels[j]=unique[0]
    if pure:
        # Exact class-stratified assignment at GROUP level, including very rare classes.
        rng=np.random.default_rng(seed);allocation=np.full(len(groups),-1,dtype=np.int64)
        class_load=np.zeros((8,k),dtype=np.int64);total_load=np.zeros(k,dtype=np.int64)
        for c in np.argsort([np.sum(labels==j) for j in range(8)]):
            ids=np.flatnonzero(labels==c);rng.shuffle(ids)
            ids=ids[np.argsort(-sizes[ids],kind='stable')]
            for j in ids:
                tie=rng.random(k)*1e-6
                f=min(range(k),key=lambda f:(class_load[c,f],total_load[f],tie[f]))
                allocation[j]=f;class_load[c,f]+=sizes[j];total_load[f]+=sizes[j]
        folds=[(np.flatnonzero(allocation[inv]!=i),np.flatnonzero(allocation[inv]==i)) for i in range(k)]
        mode='CLASS_BALANCED_GROUP_ASSIGNMENT'
    else:
        # Groups with mixed diagnoses require an iterative stratifier. Only accept valid folds.
        folds=[];mode='STRATIFIED_GROUP_KFOLD_RETRY'
        for n in range(k,1,-1):
            for attempt in range(200):
                candidate=list(StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=seed+attempt)
                               .split(np.zeros(len(y)),y,g))
                if all(len(np.unique(y[tr]))==8 and len(np.unique(y[te]))==8 for tr,te in candidate):
                    folds=candidate;break
            if folds:break
        if not folds:
            return [],{'mode':'NO_CLASS_COMPLETE_GROUP_CV','folds':0,'group_support':support,
                       'reason':'Mixed-label groups: no class-complete lesion-disjoint CV split found after 200 attempts per fold count.'}
    check=np.zeros(len(y),dtype=np.int64)
    for tr,te in folds:
        req(len(tr)>0 and len(te)>0 and set(g[tr]).isdisjoint(set(g[te])),'Lesion crossed CV split')
        req(len(np.unique(y[tr]))==8 and len(np.unique(y[te]))==8,'CV missing a diagnostic class')
        check[te]+=1
    req(np.all(check==1),'Each meta-training sample must be held out exactly once in grouped CV')
    return folds,{'mode':mode,'folds':len(folds),'group_support':support,
                  'fold_sizes':[int(len(te)) for _,te in folds]}

cv_meta,CV_INFO=safe_group_cv(y_meta,g_meta,FOLDS,SEED)
print('META TRAINING grouped-CV audit:',json.dumps(CV_INFO))
print('VALIDATION group support:',group_support(y_val,g_val),'— evaluated as one complete held-out partition')
jwrite(OUT/'reports/group_cv_audit.json',{'meta_train':CV_INFO,'validation_group_support':group_support(y_val,g_val),'validation_evaluated_as_full_partition':True})

# Temperatures are fitted ONLY on meta_train; meta CV refits them within every training fold.
temps={s:calibrate(prob['meta_train'][s],y_meta) for s in SOURCES}
cal={part:{s:tscale(prob[part][s],temps[s]) for s in SOURCES} for part in prob}
calrows=[{'source':s,'temperature':temps[s],
          'meta_nll_before':score(y_meta,prob['meta_train'][s])['nll'],
          'meta_nll_after':score(y_meta,cal['meta_train'][s])['nll'],
          'validation_bacc':score(y_val,prob['validation'][s])['balanced_accuracy']} for s in SOURCES]
pd.DataFrame(calrows).to_csv(OUT/'tables/temperature_scaling.csv',index=False)
print(pd.DataFrame(calrows).round(5).to_string(index=False))

def features(d):return np.concatenate([np.log(np.clip(d[s],1e-12,1.)) for s in SOURCES],axis=1)
def make_stack(C):
    return Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(
        C=C,class_weight='balanced',solver='lbfgs',max_iter=3000,random_state=SEED))])
def aligned(model,x):
    z=model.predict_proba(x);ids=model.named_steps['clf'].classes_.astype(int)
    req(np.array_equal(ids,np.arange(8)),'Stacker is missing a diagnostic class')
    return normalize(z)
def complete_fold_stats(y,p,cv):
    req(len(cv)>0,'No class-complete CV available')
    pred=np.argmax(p,axis=1)
    vals=np.array([balanced_accuracy_score(y[te],pred[te]) for _,te in cv])
    return {'cv_mean_bacc':float(vals.mean()),'cv_sd_bacc':float(vals.std()),
            'cv_min_bacc':float(vals.min()),'cv_pooled_bacc':float(balanced_accuracy_score(y,pred))}

META TRAINING grouped-CV audit: {"mode": "CLASS_BALANCED_GROUP_ASSIGNMENT", "folds": 5, "group_support": {"MEL": 132, "NV": 717, "BCC": 142, "AK": 33, "BKL": 135, "DF": 12, "VASC": 20, "SCC": 24}, "fold_sizes": [453, 454, 454, 453, 453]}
VALIDATION group support: {'MEL': 117, 'NV': 764, 'BCC': 117, 'AK': 33, 'BKL': 138, 'DF': 6, 'VASC': 17, 'SCC': 24} — evaluated as one complete held-out partition
        source  temperature  meta_nll_before  meta_nll_after  validation_bacc
      rotation      1.42674          0.89729         0.85214          0.57717
        simclr      2.48793          1.28027         0.91843          0.57562
  FusionConcat      1.84106          1.06732         0.87941          0.58140
 FusionGatedV2      2.07582          1.29016         0.94753          0.58359
 FusionGatedV3      1.84788          1.15235         0.91418          0.57827
ResidualFusion      1.95072          1.21350         0.92594          0.60185


## 2. Candidate registration, evaluation and reference ensembles

In [5]:
# All base model probabilities are already aligned by image ID, class order, group and hash.
S=len(SOURCES);T=len(y_meta);V=len(y_val)
meta_p={s:cal['meta_train'][s] for s in SOURCES};val_p={s:cal['validation'][s] for s in SOURCES}
raw_meta={s:prob['meta_train'][s] for s in SOURCES};raw_val={s:prob['validation'][s] for s in SOURCES}
registry={};models={};meta_records=[]

def add(name,family,mp,vp,spec,meta_selected=False):
    req(name not in registry,f'Duplicate candidate {name}')
    mp=normalize(mp);vp=normalize(vp)
    req(mp.shape==(T,8) and vp.shape==(V,8),f'Candidate shape mismatch {name}')
    registry[name]={'family':family,'spec':spec,'p':vp,'meta':mp,'meta_selected':meta_selected}
    return name

def fuse(d,keys,weights=None,mode='arithmetic'):
    p=np.stack([d[s] for s in keys],axis=0)
    w=np.full(len(keys),1/len(keys)) if weights is None else np.asarray(weights,dtype=np.float64)
    req(len(w)==len(keys) and np.all(w>=0) and np.isclose(w.sum(),1),'Invalid weights')
    if mode=='arithmetic':return normalize(np.einsum('snc,s->nc',p,w,optimize=True))
    if mode=='geometric':return softmax(np.einsum('snc,s->nc',np.log(np.clip(p,1e-12,1)),w,optimize=True),axis=1)
    if mode=='vote':
        votes=np.eye(8,dtype=np.float64)[p.argmax(axis=2)]
        return normalize(np.einsum('snc,s->nc',votes,w,optimize=True)+.0001*np.einsum('snc,s->nc',p,w,optimize=True))
    raise ValueError(mode)

for s in SOURCES:
    add('Base_'+s,'Single model',raw_meta[s],raw_val[s],{'kind':'base','source':s,'calibrated':False})
    add('Cal_'+s,'Calibrated single',meta_p[s],val_p[s],{'kind':'base','source':s,'calibrated':True})
PAIRS={'SSL':['rotation','simclr'],'SimCLR_Residual':['simclr','ResidualFusion'],
       'Fusion4':SOURCES[2:],'All6':SOURCES}
for key,subset in PAIRS.items():
    for mode in ['arithmetic','geometric','vote']:
        add(f'{mode}_{key}','Simple '+mode,fuse(meta_p,subset,mode=mode),fuse(val_p,subset,mode=mode),
            {'kind':mode,'sources':subset,'weights':[1/len(subset)]*len(subset)})
print('Reference candidates',len(registry),'| Baseline R06 ResidualFusion BACC',round(100*score(y_val,raw_val['ResidualFusion'])['balanced_accuracy'],3))

Reference candidates 24 | Baseline R06 ResidualFusion BACC 60.185


## 3. Thesis-style constrained weighted ensembles (fit on meta-train only)

In [6]:
# Candidate grid inspired by the original thesis: pairwise, Dirichlet, SimCLR-dominant, all-source.
# Weight ranking is meta_train optimization, not unbiased CV: fold dispersion is a robustness diagnostic.
rng=np.random.default_rng(SEED+91)
weight_groups={
 'AllConstrained':np.vstack([np.eye(S),np.full((1,S),1/S)]+[
    rng.dirichlet(np.full(S,a),size=max(256,WEIGHT_SAMPLES//3)) for a in [.35,1.0,2.5]]),
 'SimCLRDominant':np.vstack([rng.dirichlet(np.array([.7,5.,.6,.6,.6,1.2]),size=WEIGHT_SAMPLES//2),
    rng.dirichlet(np.array([1.0,4.0,1.,1.,1.,2.]),size=WEIGHT_SAMPLES//2)]),
 'ResidualDominant':np.vstack([rng.dirichlet(np.array([.7,1.3,.6,.6,.6,5.]),size=WEIGHT_SAMPLES//2),
    rng.dirichlet(np.array([1.,2.,1.,1.,1.,4.]),size=WEIGHT_SAMPLES//2)])}
for group in ['SimCLRDominant','ResidualDominant']:
    anchor='simclr' if group.startswith('SimCLR') else 'ResidualFusion'
    i=SOURCES.index(anchor)
    weight_groups[group]=weight_groups[group][weight_groups[group][:,i]>=.4]
# Deterministic two-model grid, and a historical SimCLR-preserving six-source constraint.
W2=[]
for a,b in [('rotation','simclr'),('simclr','ResidualFusion'),('rotation','ResidualFusion')]:
    for w in np.linspace(.05,.95,19):
        vec=np.zeros(S);vec[SOURCES.index(a)]=w;vec[SOURCES.index(b)]=1-w;W2.append(vec)
weight_groups['PairwiseGrid']=np.asarray(W2)
for group in weight_groups:
    W=np.asarray(weight_groups[group],dtype=np.float32);W=W[np.isclose(W.sum(1),1.)]
    req(len(W)>0,f'No valid weights for {group}')
    weight_groups[group]=W

meta_stack=np.stack([meta_p[s] for s in SOURCES],axis=0).astype('float32')
val_stack=np.stack([val_p[s] for s in SOURCES],axis=0).astype('float32')
wm=[];selected_weights={}
for family,W in weight_groups.items():
    rows=[]
    for start in range(0,len(W),64):
        w=W[start:start+64]
        preds=np.einsum('snc,ks->knc',meta_stack,w,optimize=True).argmax(2)
        recall=np.stack([(preds[:,y_meta==c]==c).mean(1) for c in range(8)])
        bacc=recall.mean(0)
        if cv_meta:
            # Every validation fold is class-complete, checked above.
            fold=np.stack([np.stack([(preds[:,te[y_meta[te]==c]]==c).mean(1)
                        for c in range(8)]).mean(0) for _,te in cv_meta],axis=1)
            sd=fold.std(1)
        else:sd=np.zeros(len(w))
        for j in range(len(w)):
            rows.append({'idx':start+j,'meta_bacc':float(bacc[j]),'fold_sd':float(sd[j]),
                         'score':float(bacc[j]-.20*sd[j])})
    frame=pd.DataFrame(rows).sort_values(['score','meta_bacc','fold_sd'],ascending=[False,False,True])
    kept=[]
    for item in frame.itertuples(index=False):
        w=W[int(item.idx)]
        if all(np.max(abs(w-other))>.06 for other in kept):
            kept.append(w.copy())
            name=f'Weighted_{family}_{len(kept)}'
            add(name,'Weighted '+family,fuse(meta_p,SOURCES,w),fuse(val_p,SOURCES,w),
                {'kind':'weighted','weights':w.astype(float).tolist(),'sources':SOURCES},True)
            selected_weights[name]=w.astype(float).tolist()
            if len(kept)==3:break
    for v in frame.head(40).itertuples(index=False):
        wm.append({'family':family,'candidate_index':int(v.idx),
                   'meta_bacc':v.meta_bacc,'fold_sd':v.fold_sd,'robust_score':v.score,
                   **{'w_'+s:float(W[int(v.idx),i]) for i,s in enumerate(SOURCES)}})
    print(family,'top meta BACC',round(100*frame.iloc[0].meta_bacc,2),'candidates',len(W))
pd.DataFrame(wm).to_csv(OUT/'tables/weighted_meta_search_top40.csv',index=False)
jwrite(OUT/'reports/weighted_selected.json',selected_weights)
print('Weighted candidates',len(selected_weights))

AllConstrained top meta BACC 58.04 candidates 4102
SimCLRDominant top meta BACC 56.49 candidates 2754
ResidualDominant top meta BACC 57.82 candidates 2712
PairwiseGrid top meta BACC 57.97 candidates 57
Weighted candidates 12


## 4. Adaptive confidence, entropy and margin gates (lesion-aware Phase-2G-style extensions)

In [7]:
# Per-image reliability weights. Hyperparameters selected on meta_train; validation is held out.
def gate(d,subset,measure,beta,prior=None):
    p=np.stack([d[s] for s in subset],axis=1)  # [n,s,8]
    if measure=='confidence':q=p.max(2)
    elif measure=='entropy':q=1.+np.sum(p*np.log(np.clip(p,1e-12,1)),axis=2)/np.log(8)
    elif measure=='margin':
        top=np.partition(p,-2,axis=2)[:,:,-2:];q=top.max(2)-top.min(2)
    else:raise ValueError(measure)
    w=softmax(beta*q,axis=1) if prior is None else softmax(beta*q+np.log(np.asarray(prior)[None,:]),axis=1)
    return normalize(np.einsum('nsc,ns->nc',p,w,optimize=True))

pruned={name:subset for name,subset in PAIRS.items() if name in ['SSL','SimCLR_Residual','All6']}
gaterows=[]
for family,subset in pruned.items():
    for measure in ['confidence','entropy','margin']:
        trials=[]
        for beta in [0.,1.,2.,4.,8.,16.]:
            p=gate(meta_p,subset,measure,beta)
            met=score(y_meta,p)
            trials.append({'beta':beta,'meta_bacc':met['balanced_accuracy'],'meta_macro_f1':met['macro_f1']})
        trials.sort(key=lambda v:(v['meta_bacc'],v['meta_macro_f1'],-v['beta']),reverse=True)
        for rank,info in enumerate(trials[:2]):
            beta=info['beta'];name=f'Gate_{measure}_{family}_{rank+1}'
            add(name,'Adaptive gate',gate(meta_p,subset,measure,beta),gate(val_p,subset,measure,beta),
                {'kind':'gate','measure':measure,'subset':subset,'beta':beta},True)
        for item in trials:gaterows.append({'family':family,'measure':measure,**item})
pd.DataFrame(gaterows).to_csv(OUT/'tables/adaptive_gate_meta_search.csv',index=False)
print('Adaptive gates added',sum(v['family']=='Adaptive gate' for v in registry.values()))

Adaptive gates added 18


## 5. Logistic stacking: raw, log-probability, log-plus-entropy; balanced/unbalanced

In [8]:
# Meta-train group-OOF CV is used to choose C when feasible; otherwise use C=0.1.
# Fold-specific temperature fits prevent held-out group labels from influencing OOF calibration.
def stackfeat(d,mode):
    x=np.concatenate([np.log(np.clip(d[s],1e-12,1)) if mode!='raw' else d[s] for s in SOURCES],axis=1)
    if mode=='extended':
        stat=[]
        for s in SOURCES:
            p=d[s];t=np.partition(p,-2,axis=1)[:,-2:]
            stat.append(np.stack([p.max(1),t.max(1)-t.min(1),-np.sum(p*np.log(np.clip(p,1e-12,1)),axis=1)/np.log(8)],axis=1))
        x=np.concatenate([x]+stat,axis=1)
    return x

def stackmodel(C,balanced):
    return Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(
        C=C,class_weight='balanced' if balanced else None,solver='lbfgs',max_iter=2500,random_state=SEED))])

def predict_aligned(m,x):
    ids=m.named_steps['clf'].classes_.astype(int)
    req(np.array_equal(ids,np.arange(8)),'Stacker trained without all classes')
    return normalize(m.predict_proba(x))

fold_cache=[]
if cv_meta:
    for tr,va in cv_meta:
        ft={s:calibrate(raw_meta[s][tr],y_meta[tr]) for s in SOURCES}
        mt={s:tscale(raw_meta[s][tr],ft[s]) for s in SOURCES}
        vt={s:tscale(raw_meta[s][va],ft[s]) for s in SOURCES}
        fold_cache.append((tr,va,mt,vt))

stack_cv=[];stack_winners={}
for mode in ['raw','log','extended']:
    for balanced in [False,True]:
        trials=[]
        for regularization in [.01,.1,1.,10.]:
            if cv_meta:
                oof=np.zeros((T,8))
                for tr,va,mt,vt in fold_cache:
                    m=stackmodel(regularization,balanced)
                    m.fit(stackfeat(mt,mode),y_meta[tr]);oof[va]=predict_aligned(m,stackfeat(vt,mode))
                req(np.allclose(oof.sum(1),1,atol=1e-5),'Incomplete stacking OOF')
                met=score(y_meta,oof)
                folds=np.array([balanced_accuracy_score(y_meta[va],oof[va].argmax(1)) for _,va in cv_meta])
                obj=met['balanced_accuracy']-.2*folds.std()
                trials.append({'C':regularization,'oof_bacc':met['balanced_accuracy'],
                               'oof_macro_f1':met['macro_f1'],'fold_sd':float(folds.std()),'score':obj})
            else:
                trials.append({'C':regularization,'oof_bacc':np.nan,'oof_macro_f1':np.nan,
                               'fold_sd':np.nan,'score':np.nan})
        if cv_meta:
            trials.sort(key=lambda q:(q['score'],q['oof_macro_f1'],-q['C']),reverse=True)
            Csel=trials[0]['C'];why='GROUP_OOF'
        else:Csel=.1;why='PREDECLARED_NO_ALLCLASS_GROUP_CV'
        name=f'Stack_{mode}_{"balanced" if balanced else "unweighted"}'
        model=stackmodel(Csel,balanced);model.fit(stackfeat(meta_p,mode),y_meta)
        pmeta=predict_aligned(model,stackfeat(meta_p,mode))
        pval=predict_aligned(model,stackfeat(val_p,mode))
        add(name,'Logistic stacking',pmeta,pval,
            {'kind':'stack','name':name,'feature_mode':mode,'balanced':balanced,'C':Csel,'selection':why},bool(cv_meta))
        models[name]=model;joblib.dump(model,OUT/'models'/f'{name}.joblib',compress=3)
        stack_winners[name]=dict(C=Csel,mode=mode,balanced=balanced,selection=why)
        for row in trials:stack_cv.append({'stack':name,**row,'selected_C':Csel,'selection':why})
        print(name,'C',Csel,'val BACC',round(100*score(y_val,pval)['balanced_accuracy'],2))
pd.DataFrame(stack_cv).to_csv(OUT/'tables/stacking_meta_group_cv.csv',index=False)
jwrite(OUT/'reports/stack_configurations.json',stack_winners)

Stack_raw_unweighted C 0.1 val BACC 53.08
Stack_raw_balanced C 0.01 val BACC 60.07
Stack_log_unweighted C 0.1 val BACC 52.26
Stack_log_balanced C 0.01 val BACC 57.93
Stack_extended_unweighted C 1.0 val BACC 51.81
Stack_extended_balanced C 0.01 val BACC 58.11


## 6. Original-thesis anchor blending and additional hybrid blends

In [9]:
# Historical proposed family: calibrated SimCLR anchor + logistic stacking + optional class bias.
# Additional anchor variants test whether ResidualFusion or weighted averaging serves better.
anchors=['Cal_simclr','Cal_ResidualFusion','arithmetic_SSL','arithmetic_All6']
weights=[n for n in registry if n.startswith('Weighted_AllConstrained_') or n.startswith('Weighted_SimCLRDominant_')]
anchors+=weights[:2]
for stackname in list(stack_winners):
    for anchor in anchors:
        for alpha in [.2,.4,.6,.8]:
            name=f'Blend_{stackname}__{anchor}__a{alpha:g}'
            a=registry[anchor];s=registry[stackname]
            add(name,'Anchored stacking',(1-alpha)*a['meta']+alpha*s['meta'],
                (1-alpha)*a['p']+alpha*s['p'],
                {'kind':'blend','stack':stackname,'anchor':anchor,'alpha':alpha},True)
print('Total pre-bias candidates:',len(registry))

Total pre-bias candidates: 204


## 7. Validation leaderboard and thesis proposed class-bias variants

In [10]:
# Report all methods, not just the winner. Per-class recalls always use the FULL validation set.
def leaderboard():
    rows=[]
    for name,item in registry.items():
        metric=score(y_val,item['p'])
        rows.append({'method':name,'family':item['family'],'meta_selected':item['meta_selected'],
                     **metric,'spec':json.dumps(item['spec'],sort_keys=True)})
    return pd.DataFrame(rows).sort_values(['balanced_accuracy','macro_f1','nll'],ascending=[False,False,True]).reset_index(drop=True)

pre=leaderboard()
pre.to_csv(OUT/'tables/all_ensembles_validation_prebias.csv',index=False)
print('Top 20 before bias tuning:')
print(pre[['method','family','balanced_accuracy','accuracy','macro_f1','nll']].head(20).round(5).to_string(index=False))

# Historical class-bias family: fine-grained validation-selected offsets, with explicit optimism warning.
# Apply to the strongest anchored stack and to the pre-bias overall leader only.
blendtop=pre[pre.family.eq('Anchored stacking')]
parents=[]
if len(blendtop):parents.append(str(blendtop.iloc[0].method))
parents.append(str(pre.iloc[0].method));parents=list(dict.fromkeys(parents))
BIAS_CANDIDATES=[-.15,-.075,0.,.075,.15]
for base_name in parents:
    base=registry[base_name];p=base['p'];b=np.zeros(8)
    base_bacc=score(y_val,p)['balanced_accuracy']
    def evaluate_bias(v):
        z=score(y_val,apply_bias(p,v))
        return (z['balanced_accuracy']-.003*np.mean(np.abs(v)),z['macro_f1'],-z['nll'])
    for _ in range(2):
        for c in range(8):
            opts=[]
            for v in BIAS_CANDIDATES:
                trial=b.copy();trial[c]=v
                opts.append((evaluate_bias(trial),-abs(v),v))
            b[c]=max(opts,key=lambda t:(t[0],t[1]))[2]
    val=apply_bias(p,b)
    gain=score(y_val,val)['balanced_accuracy']-base_bacc
    if gain>=.003:
        add('ClassBias_'+base_name,'Thesis class-bias',apply_bias(base['meta'],b),val,
            {'kind':'class_bias','parent':base_name,'offsets':b.tolist(),
             'selection':'VALIDATION_OPTIMISTIC'},False)
    print('Class-bias',base_name,'gain_pp',round(gain*100,3),'accepted',gain>=.003,'offsets',b)

rank=leaderboard();rank.to_csv(OUT/'tables/all_ensembles_validation.csv',index=False)
selected=str(rank.iloc[0].method);winner=registry[selected]
# One best representative per family, for transparent thesis ablation.
family_rank=rank.groupby('family',sort=False).head(1).reset_index(drop=True)
family_rank.to_csv(OUT/'tables/best_by_family.csv',index=False)
print('\nBest per family:\n',family_rank[['method','family','balanced_accuracy','accuracy','macro_f1']].round(5).to_string(index=False))
print('\nWinner:',selected,'validation BACC',round(100*score(y_val,winner['p'])['balanced_accuracy'],3))

Top 20 before bias tuning:
                                                        method                  family  balanced_accuracy  accuracy  macro_f1     nll
Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4       Anchored stacking            0.61691   0.71500   0.54126 0.81431
Blend_Stack_extended_balanced__Weighted_AllConstrained_2__a0.4       Anchored stacking            0.61651   0.71409   0.54284 0.81280
Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.2       Anchored stacking            0.61117   0.72049   0.54796 0.78590
     Blend_Stack_log_balanced__Weighted_AllConstrained_1__a0.4       Anchored stacking            0.61115   0.71409   0.53896 0.81706
     Blend_Stack_log_balanced__Weighted_AllConstrained_1__a0.2       Anchored stacking            0.61090   0.72049   0.54769 0.78719
            Blend_Stack_raw_balanced__Cal_ResidualFusion__a0.6       Anchored stacking            0.60839   0.71363   0.53729 0.90258
                                   

## 8. Final artifact, completeness checks, and export for locked R08 evaluation

In [11]:
final=winner['p'];prediction=final.argmax(1);result=score(y_val,final)
baseline=score(y_val,raw_val['ResidualFusion'])['balanced_accuracy']
cm=confusion_matrix(y_val,prediction,labels=np.arange(8))
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'tables/final_validation_confusion_matrix.csv')
pd.DataFrame(classification_report(y_val,prediction,labels=np.arange(8),target_names=CLASSES,
                     output_dict=True,zero_division=0)).T.to_csv(OUT/'tables/final_validation_classification_report.csv')
np.savez_compressed(OUT/'predictions/final_validation.npz',
    image_id=meta['validation'].image_id.astype(str).to_numpy(),group_id=g_val,
    y=y_val,proba=final.astype('float32'),classes=np.array(CLASSES),method=selected)
for s in SOURCES:
    np.savez_compressed(OUT/'predictions'/f'{s}_meta_val.npz',
        meta_image_id=meta['meta_train'].image_id.astype(str).to_numpy(),
        val_image_id=meta['validation'].image_id.astype(str).to_numpy(),meta_y=y_meta,val_y=y_val,
        meta_proba=raw_meta[s].astype('float32'),val_proba=raw_val[s].astype('float32'))

# Full candidate probabilities facilitate exact post-hoc auditing; do not use to rerank on test.
for name,item in registry.items():
    # Save individually to avoid a large dense 3D array and preserve exact name/spec.
    np.save(OUT/'predictions'/f'candidate_{name}.npy',item['p'].astype('float32'))

config={'name':NAME,'stage':'R07','seed':SEED,'status':'COMPLETED',
        'classes':CLASSES,'sources':SOURCES,'n01_manifest_sha256':s05['n01_manifest_sha256'],
        'r04_encoder_provenance':s05['r04_provenance'],'r06_status_sha256':sha(R06/'reports/final_status.json'),
        'temperature_fit':'meta_train','stacker_fit':'meta_train','final_selection':'validation',
        'meta_group_cv':CV_INFO,'temperatures':temps,'models':stack_winners,
        'selected_method':selected,'selected_spec':winner['spec'],'all_candidate_specs':{k:v['spec'] for k,v in registry.items()},
        'validation_metrics':result,'residualfusion_bacc':baseline,
        'validation_delta_bacc_vs_residualfusion':result['balanced_accuracy']-baseline,
        'validation_search_is_exploratory':True,'test_touched':False,
        'notes':'Candidate-selection and class bias examined validation; validation score is optimistic. Freeze configuration; evaluate locked N01 test in separate R08.'}
canon=json.dumps(config,sort_keys=True,default=str,allow_nan=False)
config['frozen_config_sha256']=hashlib.sha256(canon.encode()).hexdigest()
jwrite(OUT/'reports/final_ensemble_config.json',config)
jwrite(OUT/'reports/final_status.json',{'stage':'R07','name':NAME,'seed':SEED,'status':'COMPLETED',
    'selected_method':selected,'test_touched':False,'n01_manifest_sha256':s05['n01_manifest_sha256'],
    'frozen_config_sha256':config['frozen_config_sha256'],'validation_metrics':result,'candidate_count':len(registry)})

req(len(rank)==len(registry) and rank.method.is_unique,'Candidate leaderboard incomplete')
req(np.isfinite(rank[['balanced_accuracy','accuracy','macro_f1','nll']].to_numpy()).all(),'NaN metrics')
req((OUT/'reports/final_ensemble_config.json').is_file(),'Frozen config missing')
archive=OUT.parent/(NAME+'.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,arcname=NAME+'/'+str(p.relative_to(OUT)))
print('R07-All COMPLETE: sources',len(SOURCES),'candidates',len(registry),'groupCV',CV_INFO['mode'])
print('WINNER:',selected,'BACC %.3f%% Accuracy %.3f%% MacroF1 %.3f%%'%(100*result['balanced_accuracy'],100*result['accuracy'],100*result['macro_f1']))
print('R06 ResidualFusion reference %.3f%% | difference %+.3f pp'%(100*baseline,100*(result['balanced_accuracy']-baseline)))
print('Frozen config:',OUT/'reports/final_ensemble_config.json','\nResults ZIP:',archive)

R07-All COMPLETE: sources 6 candidates 205 groupCV CLASS_BALANCED_GROUP_ASSIGNMENT
WINNER: ClassBias_Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4 BACC 62.044% Accuracy 71.866% MacroF1 54.724%
R06 ResidualFusion reference 60.185% | difference +1.860 pp
Frozen config: /kaggle/working/lesion-r07-n01-all-ensembles-s42/reports/final_ensemble_config.json 
Results ZIP: /kaggle/working/lesion-r07-n01-all-ensembles-s42.zip
